In [ ]:
import json
from collections import Counter
from pathlib import Path

import nltk
import pandas as pd
import requests
import spacy
from bs4 import BeautifulSoup

# Optional one-time lightweight setup checks for the NLP libraries.
nltk.download("punkt", quiet=True)
_ = spacy.blank("en")

SITES = {
    "TechCrunch": "https://techcrunch.com/feed/",
    "The Verge": "https://www.theverge.com/rss/index.xml",
    "Ars Technica": "https://feeds.arstechnica.com/arstechnica/index",
}


def clean_html(text: str) -> str:
    if not text:
        return ""
    return BeautifulSoup(text, "html.parser").get_text(" ", strip=True)


def first_text(node, tag_names: list[str]) -> str:
    for name in tag_names:
        found = node.find(name)
        if found and found.get_text(strip=True):
            return found.get_text(strip=True)
    return ""


def extract_link(node) -> str:
    # RSS feeds often use <link>url</link>, while Atom uses <link href="..." />.
    link_tag = node.find("link")
    if not link_tag:
        return ""

    href = link_tag.get("href")
    if href:
        return href.strip()

    link_text = link_tag.get_text(strip=True)
    return link_text.strip() if link_text else ""


def fetch_feed_articles(source: str, feed_url: str, limit: int = 20) -> list[dict]:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
    }
    response = requests.get(feed_url, headers=headers, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "xml")

    # Support both RSS (<item>) and Atom (<entry>) formats.
    entries = soup.find_all("item")
    if not entries:
        entries = soup.find_all("entry")

    articles = []
    for entry in entries[:limit]:
        headline = first_text(entry, ["title"])
        article_url = extract_link(entry)

        summary_raw = first_text(
            entry,
            ["content:encoded", "content", "summary", "description"],
        )
        article_summary = clean_html(summary_raw)

        if headline and article_url:
            articles.append(
                {
                    "source": source,
                    "headline": headline,
                    "url": article_url,
                    "summary": article_summary,
                }
            )

    return articles


all_articles = []
errors = {}

for source_name, feed in SITES.items():
    try:
        site_articles = fetch_feed_articles(source_name, feed, limit=20)
        all_articles.extend(site_articles)
    except Exception as exc:
        errors[source_name] = str(exc)

output_dir = Path.cwd()
csv_path = output_dir / "tech_news_articles.csv"
json_path = output_dir / "tech_news_articles.json"

if all_articles:
    df = pd.DataFrame(all_articles)
    df.to_csv(csv_path, index=False, encoding="utf-8")

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_articles, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(all_articles)} articles to:")
    print(f"- CSV:  {csv_path}")
    print(f"- JSON: {json_path}")

    print("\nTotal number of articles collected from each website:")
    counts = Counter(article["source"] for article in all_articles)
    for site_name in SITES:
        print(f"- {site_name}: {counts.get(site_name, 0)}")
else:
    print("No articles were collected.")

if errors:
    print("\nWebsites with errors:")
    for site_name, message in errors.items():
        print(f"- {site_name}: {message}")

Saved 40 articles to:
- CSV:  c:\Users\maheit\Desktop\tech_news_articles.csv
- JSON: c:\Users\maheit\Desktop\tech_news_articles.json

Total number of articles collected from each website:
- TechCrunch: 20
- The Verge: 0
- Ars Technica: 20
